# Ch.2 座学：データ可視化・EDA
### 講義時間：20分

---

> **講師メモ**：「グラフを作ること」より「グラフから何を読み取るか」に焦点を当てる。  
> 各グラフを実行したあと、必ず「これを見てどう思いますか？」と問いかける。  
> 受講者が「気づき」を言語化する練習をさせることがこの章の最大の目的。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_wine

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('データ準備完了')

---
## 【スライド1】EDA（探索的データ分析）とは

### Exploratory Data Analysis

> **モデルを作る前に、データの特徴・傾向・外れ値・相関を可視化によって把握する作業**

### なぜ EDA が重要か

```
良いモデルを作る = 良いデータ理解から始まる
```

| EDA をしないと… | EDA をすると… |
|----------------|---------------|
| データの異常に気づかない | 欠損・外れ値を早期発見できる |
| 重要でない変数にも同じ手間をかける | 効果的な変数を絞り込める |
| モデルの結果が「なぜか」わからない | 「この変数が効いていそう」と仮説が立てられる |

### 実務での位置づけ
データサイエンティストの業務時間の **20〜30%** を EDA が占めると言われています。

---
## 【スライド2】グラフの種類と使い分け

### 「何を見たいか」でグラフを選ぶ

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis('off')

items = [
    ('ヒストグラム\n`hist()`', '1変数の\n分布を見る', '#4A90D9'),
    ('箱ひげ図\n`boxplot()`', 'グループ間の\n分布を比較', '#5BA85A'),
    ('散布図\n`scatter()`', '2変数の\n関係を見る', '#E8A838'),
    ('ヒートマップ\n`heatmap()`', '多変数の\n相関を一覧', '#C0392B'),
    ('棒グラフ\n`bar()`', 'グループ間の\n値を比較', '#8E44AD'),
]

for i, (name, desc, color) in enumerate(items):
    x = 0.02 + i * 0.198
    box = mpatches.FancyBboxPatch((x, 0.1), 0.175, 0.82,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='white', lw=2,
        transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x + 0.0875, 0.80, name, transform=ax.transAxes,
            ha='center', va='top', fontsize=10, color='white', fontweight='bold')
    ax.text(x + 0.0875, 0.35, desc, transform=ax.transAxes,
            ha='center', va='center', fontsize=9.5, color='#FDFEFE')

ax.set_title('グラフの種類と用途', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

---
## 【スライド3】ヒストグラム：データの分布を見る

### 「この変数の値はどんな形（分布）をしているか？」を確認

In [ ]:
# デモ：アルコール度数のヒストグラム（全体 vs 品種別）
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：全体
axes[0].hist(df['alcohol'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('全体の分布')
axes[0].set_xlabel('アルコール度数')
axes[0].set_ylabel('件数')
axes[0].axvline(df['alcohol'].mean(), color='red', linestyle='--', label=f'平均: {df["alcohol"].mean():.2f}')
axes[0].legend()

# 右：品種別
colors = {'Barolo': '#e74c3c', 'Grignolino': '#2ecc71', 'Barbera': '#3498db'}
for name, group in df.groupby('品種'):
    axes[1].hist(group['alcohol'], bins=15, alpha=0.6, label=name,
                 color=colors[name], edgecolor='white')
axes[1].set_title('品種別の分布')
axes[1].set_xlabel('アルコール度数')
axes[1].set_ylabel('件数')
axes[1].legend()

plt.suptitle('ヒストグラム：アルコール度数の分布', fontsize=13)
plt.tight_layout()
plt.show()

print('観察ポイント：Barolo と他の2品種でアルコール度数の分布が違う。')
print('→ 品種の分類に「アルコール度数」が使えそう！（仮説）')

---
## 【スライド4】箱ひげ図：グループ間を比較する

### 箱ひげ図の読み方

In [ ]:
# 箱ひげ図の読み方を説明する図
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：読み方の説明図
ax = axes[0]
sample_data = [7, 8, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 15, 18]
bp = ax.boxplot(sample_data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#AED6F1', color='#2471A3'),
                medianprops=dict(color='red', linewidth=2))

q1 = np.percentile(sample_data, 25)
q3 = np.percentile(sample_data, 75)
median = np.median(sample_data)

ax.annotate('中央値（Q2）', xy=(1.08, median), xytext=(1.3, median),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate('第3四分位数（Q3）\n上位25%の境界', xy=(1.08, q3), xytext=(1.3, q3 + 0.5),
            fontsize=8.5, color='#2471A3',
            arrowprops=dict(arrowstyle='->', color='#2471A3'))
ax.annotate('第1四分位数（Q1）\n下位25%の境界', xy=(1.08, q1), xytext=(1.3, q1 - 0.8),
            fontsize=8.5, color='#2471A3',
            arrowprops=dict(arrowstyle='->', color='#2471A3'))
ax.annotate('外れ値（ひげの外）', xy=(1.04, 18), xytext=(1.3, 17.5),
            fontsize=8.5, color='#C0392B',
            arrowprops=dict(arrowstyle='->', color='#C0392B'))
ax.set_title('箱ひげ図の読み方', fontsize=12)
ax.set_xlim(0.5, 2.2)
ax.set_xticklabels([''])

# 右：実データ
df.boxplot(column='color_intensity', by='品種', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'))
axes[1].set_title('品種ごとの色の濃さ（color_intensity）')
axes[1].set_xlabel('品種')
axes[1].set_ylabel('color_intensity')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('観察ポイント：Barolo は色が特に濃く、ばらつきも大きい。')
print('Grignolino は淡い色のワインに集中している。')

---
## 【スライド5】散布図と相関

### 「2つの変数は関係しているか？」を見る

In [ ]:
# 相関係数の概念を説明する図
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

np.random.seed(42)
x = np.linspace(0, 10, 50)

patterns = [
    (x + np.random.randn(50) * 0.5,  '正の相関\n（r ≈ +1に近い）\n一方が増えると他方も増える'),
    (-x + np.random.randn(50) * 0.5, '負の相関\n（r ≈ -1に近い）\n一方が増えると他方は減る'),
    (np.random.randn(50) * 3,         '相関なし\n（r ≈ 0）\n2変数に関係がない'),
]

colors_list = ['#e74c3c', '#3498db', '#95a5a6']
for ax, (y, title), c in zip(axes, patterns, colors_list):
    ax.scatter(x, y, alpha=0.7, color=c, s=50)
    r = np.corrcoef(x, y)[0, 1]
    ax.set_title(title + f'\nr = {r:.2f}', fontsize=10)
    ax.set_xlabel('変数 X')
    ax.set_ylabel('変数 Y')
    ax.grid(True, alpha=0.3)

plt.suptitle('相関係数（r）のパターン', fontsize=13)
plt.tight_layout()
plt.show()

print('注意：相関は「傾向」を示すが、「因果関係」ではありません。')
print('例）アイスが売れると溺死者が増える → 原因は「夏」（気温）')

In [ ]:
# デモ：ワインデータの散布図 + 相関ヒートマップ
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：散布図
colors_map = {'Barolo': '#e74c3c', 'Grignolino': '#2ecc71', 'Barbera': '#3498db'}
for name, group in df.groupby('品種'):
    axes[0].scatter(group['alcohol'], group['proline'],
                    label=name, color=colors_map[name], alpha=0.7, s=60)
axes[0].set_title('アルコール度数 vs プロリン含有量')
axes[0].set_xlabel('アルコール度数')
axes[0].set_ylabel('プロリン含有量')
axes[0].legend(title='品種')

# 右：相関ヒートマップ（主要8変数）
selected = ['alcohol', 'color_intensity', 'proline', 'flavanoids',
            'total_phenols', 'hue', 'malic_acid', 'ash']
corr = df[selected].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[1], annot_kws={'size': 8})
axes[1].set_title('相関ヒートマップ（主要変数）')

plt.suptitle('散布図と相関ヒートマップ', fontsize=13)
plt.tight_layout()
plt.show()

print('観察ポイント：品種ごとにクラスターが形成されている')
print('→ アルコール×プロリンの組み合わせで品種が区別できそう（仮説）')

---
## 【スライド6】EDA の進め方：実務フロー

### データをもらったときの標準的な手順

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

steps = [
    ('STEP 1\nデータの\n形を確認', 'head/info\ndescribe', '#4A90D9'),
    ('STEP 2\n欠損値を\n確認', 'isnull\n.sum()', '#5BA85A'),
    ('STEP 3\n分布を\n見る', 'ヒストグラム\n箱ひげ図', '#E8A838'),
    ('STEP 4\n変数間の\n関係を見る', '散布図\nヒートマップ', '#C0392B'),
    ('STEP 5\n仮説を\n言語化', '「〇〇が\n効きそう」', '#8E44AD'),
]

for i, (step, tool, color) in enumerate(steps):
    x = 0.02 + i * 0.198
    box = mpatches.FancyBboxPatch((x, 0.12), 0.17, 0.78,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='white', lw=2,
        transform=ax.transAxes)
    ax.add_patch(box)
    ax.text(x + 0.085, 0.78, step, transform=ax.transAxes,
            ha='center', va='top', fontsize=9.5, color='white', fontweight='bold')
    ax.text(x + 0.085, 0.32, tool, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, color='#FDFEFE',
            fontfamily='monospace')
    if i < 4:
        ax.annotate('', xy=(x + 0.20, 0.51), xytext=(x + 0.175, 0.51),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.set_title('EDA の標準フロー', fontsize=14, pad=10)
plt.tight_layout()
plt.show()

---
## 【スライド7】まとめ

### Ch.2 で覚えておくこと

| グラフ | コード | 使い場面 |
|--------|--------|----------|
| ヒストグラム | `plt.hist()` | 1変数の分布を確認 |
| 箱ひげ図 | `df.boxplot()` | グループ間の比較 |
| 散布図 | `plt.scatter()` | 2変数の関係確認 |
| 相関ヒートマップ | `sns.heatmap(df.corr())` | 全変数の相関を一覧 |

### EDA の本当の目的

> **「グラフを作ること」ではなく、「仮説を立てること」**
>
> グラフを見て「〇〇という変数が重要そう」「△△には関係がなさそう」という  
> 仮説を言語化することが、次のモデル構築の出発点になります。

### 次は「画像処理」へ
> 数値データの次は「画像データ」を扱います。画像の正体は、実は数値の配列です。